In [1]:
import sys

import os

In [2]:
import torch
from torch import nn
from torch.nn import functional as F


import wandb
import pandas as pd

from tqdm import tqdm

from matplotlib import pyplot as plt

import numpy as np 

from torch.utils.data import DataLoader
from xaikd.utils import metrics
from xaikd import models, datasets

from contextlib import nullcontext

from datetime import datetime

from matplotlib import colormaps

from xaikd.models.vgg import canonize_model


from zennit.attribution import Gradient
from zennit.composites import EpsilonGammaBox, EpsilonPlusFlat
from xaikd import datasets, models, utils

from torchvision import transforms

from collections import OrderedDict

In [3]:
api = wandb.Api()

In [4]:
PROJECT = "p16i/xaikd-distillation-layerwise-s27"
EXPERIMENT_GROUP = "./artifacts/2024-05-s27/experiments/4-effect-of-layerwise-losses-partition@50"

In [17]:
def read_data(
    teacher,
    student,
    dataset,
    policy,
    lambda_task,
    lambda_kd,
    lambda_layer,
    layer,
    training_size,
    contamination_level,
    parameter_partition_mode,
    verbose=False,
    return_raw = False,
    accuracy_mode="last_checkpoint",
    
):
    
    
    runs = api.runs(
        PROJECT,
        {
        "$and": [
            {"group": EXPERIMENT_GROUP},
            {"config.teacher": teacher},
            {"config.student": student},
            {"config.lambda_task": lambda_task},
            {"config.lambda_kd": lambda_kd},
            {"config.lambda_layer": lambda_layer},
            {"config.policy": policy},
            {"config.dataset": dataset},
            {"config.training_size": training_size},
            {"config.parameter_partition_mode": parameter_partition_mode},
            {"config.layers": layer},
        ]}
    )
    
    if verbose:
        print(f"we have {len(runs)} runs")
    
    rows = []
    
    count = 0
    
    datasets = []
    for run in runs:
        seed = run.config["seed"]
        if not "arr_metrics"in run.summary:
            print(">> bad run config (id={rn}")
            print(run.config)
            count += 1
            return run
            continue
                                 
        if accuracy_mode == "last_checkpoint":
            k = -1
        else:
            raise
    
        datasets.append(run.config["dataset"])

        
        best_val_acc = run.summary["arr_metrics"]["val_acc"][k]
        
        best_train_acc = run.summary["arr_metrics"]["train_acc"][k]
        


        rows.append(
            dict(
                best_epoch=k,
                seed=seed, 
                lambda_task=run.config["lambda_task"],
                lambda_kd=run.config["lambda_kd"],
                # remark: this is the actual value used in distillation
                # cf. https://github.com/p16i/xai-kd/blob/2023-09-s14--1-teacher-training-script/scripts/distill-layerwise.py#L364
                lambda_layer=run.config["policy_lambda_layer"],
                val_acc=best_val_acc, 
                train_acc=best_train_acc,
                teacher_acc=run.summary["teacher_acc"],
                policy=run.config["policy"],
                run_id=run.id
            )
        )

    if count != 0:
        print(f"We have {count} bad run! abort!")
        raise
    
    df = pd.DataFrame(rows)
    df = df.sort_values(by=["lambda_layer", "seed"], ascending=True)
        
        
    if return_raw:
        return df
        
    df = df.groupby(by="lambda_layer").agg(
        {
            "val_acc": ["mean", "std", "count"],
            "train_acc": ["mean", "std", "count"],

            "teacher_acc": ["mean"],
        }
    ).reset_index()
    
    np.testing.assert_allclose(
        df["val_acc"]["count"].values,
        3
    )
    
    if verbose:
        print(f"df.shape: {df.shape}")
    return df

summary = read_data(
    teacher="imagenet-vgg16-tv",
    dataset="imagenet-butterfly-spurious0.5",
    student="vggcustomimagenetdims-32-24-24-10",
    lambda_task=0.0,
    lambda_kd=1.0,
    lambda_layer=1.0,
    policy="basis-identity:pca--uncentered",
    training_size=0.125,
    verbose=True,
    contamination_level=0.0,
    return_raw=False,
    parameter_partition_mode="@50",
    layer="features.9:layer1,features.16:layer2,features.23:layer3,features.30:layer4"
)

summary

we have 3 runs
df.shape: (1, 8)


lambda_layer   val_acc                train_acc                 teacher_acc
                    mean      std count      mean       std count        mean
0            1  0.856667  0.01453     3  0.837792  0.004752     3    0.956667

In [18]:
def alias(name):
        __dd = {
            "basis-identity:random--uncentered": "Random Subspace",
            "basis-identity:pca--uncentered": "PCA",
            "basis-identity:pcalookahead--uncentered": "PCA-AH",
            "basis-identity:prca-sortabs--uncentered": "PRCA-sortabs",
            "vid": "VID",
            "nothing": "KD Only"

        }
        return __dd.get(name, name)
    
def color(name):
        __dd = {
            "basis-identity:random--uncentered": "lightgray"
        }
        return __dd.get(name, None)
    
def marker(name):
    if "pca" in name or "prca" in name:
        return "s"
    elif "vid" in name:
        return "x"
    else:
        return '.'

In [ ]:
def viz(dataset, lambda_layer):

    
    lambda_task = 0.0
    lambda_kd = 1.0
    parameter_partition_mode = "@50"
    teacher = "imagenet-vgg16-tv"
    student = "vggcustomimagenetdims-32-24-24-10"

    arr_layers = "features.9:layer1,features.16:layer2,features.23:layer3,features.30:layer4".split(",")
    
    arr_policies = [
        #"basis-identity:pcalookahead--uncentered",

        "basis-identity:pca--uncentered",
        "vid",
        #"basis-identity:prca-sortabs--uncentered",

        #"vid",
        #"attention-transfer",
        #"basis-identity:random--uncentered"
    ]
    
    trainining_size = 0.125
    ncols = len(arr_policies)
    nrows = 2
    num_policies = len(arr_policies)
    
    plt.figure(figsize=(2.5*ncols, 2.5*nrows))

    plt.suptitle(f"{dataset} (training_size={trainining_size}, lambda_layer={lambda_layer}, partition_update={parameter_partition_mode})\n({teacher} → {student})", y=1.1)

    ticks = np.arange(len(arr_layers))
    
    for pix, policy in enumerate(arr_policies):

        for dix, col_name in enumerate(["train_acc", "val_acc"]):
            plt.subplot(nrows, ncols, pix*ncols + dix+1)
            plt.title(f"{alias(policy)}:{col_name}")
            arr_stats = []
            arr_layer_str = []
            for lix in range(len(arr_layers)):
                layer_str = ",".join(arr_layers[lix:])
                df = read_data(
                    teacher=teacher,
                    dataset=dataset,
                    student=student,
                    lambda_task=lambda_task,
                    lambda_kd=lambda_kd,
                    lambda_layer=lambda_layer,
                    policy=policy,
                    training_size=trainining_size,
                    contamination_level=0.0,
                    layer=layer_str,
                    parameter_partition_mode=parameter_partition_mode
                )
                arr_stats.append(
                    df[col_name]["mean"].values[0]
                )
                arr_layer_str.append(layer_str)

            plt.barh(
                ticks, 
                arr_stats, 
                label=policy
            )

            
            if dix == 0:
                plt.yticks(
                    ticks,
                    arr_layer_str
                )
                                
            if col_name == "val_acc":
                teacher_acc = df["teacher_acc"].values[0] 
                plt.axvline(teacher_acc, ls="--", label="Teacher", color="k")
                plt.yticks([])
            
            if pix == 0:
                plt.xticks([])

            plt.xlim([0.5, 1])
            
viz(dataset="imagenet-butterfly-spurious0.5", lambda_layer=1.0)